# 🦀 Day 2: Architecture — Modules, Crate Structure, Dependency Planning
---

Today we design the architecture of our capstone project.

- **Workspace vs single crate**: When to use each
- **Module design**: Separation of concerns, clear APIs
- **RustKV crate structure**: `storage/`, `protocol/`, `server/`, `client/`
- **Trait-based design**: `Storage`, `Protocol` traits for flexibility
- **Dependency planning**: Which external crates and why

## Workspace vs Single Crate

| Approach | Use When | Example |
|----------|----------|--------|
| **Single crate** | Small project, one binary | RustKV (storage + server + client in one) |
| **Workspace** | Multiple binaries, shared lib, or large codebase | `rustkv` lib + `rustkv-server` + `rustkv-cli` bins |

**RustKV**: Start with single crate; split into workspace later if needed.

## RustKV Module Tree

```
rustkv/
├── src/
│   ├── lib.rs          # Re-exports, public API
│   ├── storage/        # Storage engine
│   │   ├── mod.rs
│   │   └── memory.rs   # MemoryStorage impl
│   ├── protocol/       # Command/response encoding
│   │   ├── mod.rs
│   │   └── resp.rs     # RESP-like protocol
│   ├── server/         # TCP server
│   │   └── mod.rs
│   └── client/         # CLI client (optional)
│       └── mod.rs
└── Cargo.toml
```

In [ ]:
// Define the Storage trait — abstraction over storage backends

use std::collections::HashMap;
use std::sync::Arc;
use std::time::Duration;

/// Value type (simplified for trait)
#[derive(Debug, Clone, PartialEq)]
pub enum Value {
    String(String),
    Integer(i64),
}

/// Storage backend trait
pub trait Storage: Send + Sync {
    fn get(&self, key: &str) -> Option<Value>;
    fn set(&self, key: &str, value: Value, ttl: Option<Duration>) -> ();
    fn delete(&self, key: &str) -> bool;
    fn keys(&self, pattern: &str) -> Vec<String>;
}

// Protocol trait for encoding/decoding
pub trait Protocol {
    type Command;
    type Response;
    fn decode(&self, input: &[u8]) -> Result<Self::Command, String>;
    fn encode(&self, response: &Self::Response) -> Vec<u8>;
}

println!("Traits defined. Implementations go in storage/ and protocol/ modules.");

In [ ]:
// Cargo.toml workspace setup example (for reference)

let workspace_toml = r#"
[workspace]
resolver = "2"
members = ["rustkv", "rustkv-server", "rustkv-cli"]

[workspace.package]
version = "0.1.0"
edition = "2021"
authors = ["You"]
license = "MIT"
"#;

let deps = r#"
# Key dependencies for RustKV:
# tokio = "1"           — async runtime (server)
# serde = { version = "1", features = ["derive"] } — serialization
# thiserror, anyhow    — error handling
# tracing             — logging
"#;

println!("{}", workspace_toml);
println!("{}", deps);

## 📝 Exercise: Create the Module Skeleton

Create the module structure for your project:

1. Add `mod storage;` and `mod protocol;` to `lib.rs`
2. Create `src/storage/mod.rs` with `pub trait Storage { ... }`
3. Create `src/protocol/mod.rs` with `pub trait Protocol { ... }`
4. Add `pub use storage::*` and `pub use protocol::*` in lib.rs

In [ ]:
// YOUR CODE HERE
// Simulate module structure — in a real project, these live in separate files

mod storage {
    pub trait Storage {
        fn get(&self, key: &str) -> Option<String>;
        fn set(&self, key: &str, value: String);
        // Add delete, keys as needed
    }
}

mod protocol {
    pub trait Protocol {
        fn decode(&self, input: &[u8]) -> Result<String, String>;
        fn encode(&self, response: &str) -> Vec<u8>;
    }
}

// Public API
pub use storage::Storage;
pub use protocol::Protocol;

println!("Module skeleton ready!");

## 🎯 Key Takeaways

1. Single crate for small projects; workspace when you have multiple binaries or a shared lib
2. Module design: `storage/`, `protocol/`, `server/`, `client/` — separation of concerns
3. Trait-based design (`Storage`, `Protocol`) enables swapping implementations
4. Plan dependencies: tokio, serde, thiserror, anyhow, tracing for RustKV
5. Clear public API via `pub use` in lib.rs

---
**Tomorrow:** CI/CD with GitHub Actions